# Crop Doctor — 20-Class 30-Epoch Retraining & ExecuTorch Export [Task TR2]
### High-Throughput (2x Tesla T4 GPUs), Zero-Leakage 80/20 Cluster Split, Resumable Checkpoints, ExecuTorch 0.6.0

**Settings Checklist (Kaggle Right Sidebar):**
- **Accelerator**: GPU T4 x 2 (or GPU P100)
- **Internet**: Internet on (required for weights download & ExecuTorch dependencies)
- **Input**: `crop-doctor-512` attached (private dataset uploaded from prepared 512x512 dataset)

**Task TR2 Retraining Specifications:**
1. **Epochs**: 30 epochs with Cosine Annealing learning rate schedule (`T_max = 30`).
2. **Resumable Checkpoints**: Saves checkpoint `/kaggle/working/checkpoint_epoch_*.pt` after every epoch. If preempted or restarted, automatically resumes from the latest saved epoch.
3. **Best Model Selection**: Pinned strictly to **grouped validation macro accuracy** (unweighted mean of per-class recalls).
4. **Early Stopping**: Halts training if no improvement in grouped validation macro accuracy for 8 consecutive epochs, restoring the best model checkpoint.
5. **Exact Preservation**: Seed 42, canonical dihedral pHash deduplication, within-class clustering, zero cluster leakage 80/20 split, balanced sampling with inverse-sqrt class weights, Platt temperature calibration, and 95th-percentile free energy thresholding.
6. **Swap Gates Verification**: Automatic comparison table against 15-epoch baseline (Overall ≥ 91.49%, Macro ≥ 92.17%, No class drop > 3.0%, Parity error < 1e-3).
7. **Deliverables Zip**: Exports `/kaggle/working/crop_doctor_final_30ep.zip` containing `vision.pte`, `labels.json`, `thresholds.json`, `metrics.json`, `class_counts.json`, `leakage_report.json`, `preprocess.json`, `parity_samples.json`, and `training_history.json`.


In [ ]:
import os, sys, json, zipfile, random, hashlib, shutil, math, time, platform
import concurrent.futures
from pathlib import Path
from collections import defaultdict, Counter

# ==============================================================================
# 0. CONFIGURATION & CONSTANTS (TASK TR2)
# ==============================================================================
SEED = 42
MAX_TRAIN_PER_CLASS = 640
MAX_CLASS_TOTAL = 800
HAMMING_DIST_DEFAULT = 8
HAMMING_DIST_TOMATO = 10
CONFLICT_MAX_DISTANCE = 2
IMAGE_SIZE = 512
EPOCHS = 30                  # 30-epoch run (Task TR2)
PATIENCE = 8                # Early stop if no improvement in macro acc for 8 epochs
BATCH_SIZE = 32             # Scaled for 2x T4 GPUs (16 per GPU)
NUM_WORKERS = 4             # Maximize data loading throughput
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
APP_REQUIRED_EXECUTORCH = "0.6.0"

CLASS_NAMES = [
    "rice_bacterial_leaf_blight",
    "rice_leaf_blast",
    "rice_sheath_blight",
    "rice_healthy",
    "cotton_bacterial_blight",
    "cotton_fusarium_wilt",
    "cotton_leaf_curl_virus",
    "cotton_healthy",
    "chilli_leaf_curl_virus",
    "chilli_nutrition_deficiency",
    "chilli_healthy",
    "tomato_leaf_blight",
    "tomato_yellow_leaf_curl_virus",
    "tomato_septoria_leaf_spot",
    "tomato_healthy",
    "maize_gray_leaf_spot",
    "maize_common_rust",
    "maize_northern_leaf_blight",
    "maize_healthy",
    "weeds_not_a_crop",
]
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff", ".jfif"}

random.seed(SEED)

# 15-Epoch Baseline Metrics (for Swap Gate verification)
BASELINE_METRICS = {
    "overall_accuracy": 0.9148681055155875,
    "macro_accuracy": 0.9216626251830276,
    "classes": {
        "rice_bacterial_leaf_blight": 0.975609756097561,
        "rice_leaf_blast": 0.923728813559322,
        "rice_sheath_blight": 0.9915966386554622,
        "rice_healthy": 1.0,
        "cotton_bacterial_blight": 0.9285714285714286,
        "cotton_fusarium_wilt": 0.9411764705882353,
        "cotton_leaf_curl_virus": 0.7777777777777778,
        "cotton_healthy": 0.9259259259259259,
        "chilli_leaf_curl_virus": 0.9814814814814815,
        "chilli_nutrition_deficiency": 0.9830508474576272,
        "chilli_healthy": 1.0,
        "tomato_leaf_blight": 0.7214285714285714,
        "tomato_yellow_leaf_curl_virus": 0.8220338983050848,
        "tomato_septoria_leaf_spot": 0.7226890756302521,
        "tomato_healthy": 0.96,
        "maize_gray_leaf_spot": 0.8888888888888888,
        "maize_common_rust": 0.96,
        "maize_northern_leaf_blight": 0.9292929292929293,
        "maize_healthy": 1.0,
        "weeds_not_a_crop": 1.0
    }
}

# Workspace directories
IS_KAGGLE = os.path.exists("/kaggle")
WORKING_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("benchmarks/runs_30ep/working")
RUN_DIR = Path("/kaggle/working/runs_30ep") if IS_KAGGLE else Path("benchmarks/runs_30ep")
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
ARTIFACT_DIR = RUN_DIR / "artifacts"
FINAL_ZIP_PATH = Path("/kaggle/working/crop_doctor_final_30ep.zip") if IS_KAGGLE else RUN_DIR / "crop_doctor_final_30ep.zip"

for p in [WORKING_DIR, RUN_DIR, CHECKPOINT_DIR, ARTIFACT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print(f"CROP DOCTOR - 20-CLASS 30-EPOCH RETRAINING & EXPORT (TASK TR2)")
print(f"Python: {platform.python_version()} | Platform: {platform.platform()}")
print("=" * 80)

# Check packages
try:
    import imagehash
except ImportError:
    print("Installing ImageHash...")
    os.system(f"{sys.executable} -m pip install -q ImageHash")
    import imagehash

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import torch
import torchvision
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms

print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"Hardware: {device} | Active GPUs: {gpu_count}")
for i in range(gpu_count):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 1)} GB VRAM)")

# ==============================================================================
# 1. DATASET DISCOVERY
# ==============================================================================
DATA_ROOT = None
search_roots = [
    Path("/kaggle/input/crop-doctor-512"),
    Path("/kaggle/input"),
    Path("/kaggle/working"),
    Path("C:/crop-data/crop_doctor_512"),
    Path("C:/crop-data"),
    Path("data"),
]

expanded_search = []
for r in search_roots:
    if r.exists():
        expanded_search.append(r)
        for child in r.iterdir():
            if child.is_dir():
                expanded_search.append(child)
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        expanded_search.append(grandchild)

for candidate in expanded_search:
    if all((candidate / c).is_dir() for c in CLASS_NAMES):
        DATA_ROOT = candidate
        break

if DATA_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate 20 class folders. Checked: {[str(p) for p in expanded_search[:15]]}"
    )

print(f"\n✓ Dataset found at: {DATA_ROOT}")

# ==============================================================================
# 2. MULTITHREADED INTEGRITY & FAST PREPROCESSING
# ==============================================================================
all_candidates = []
for class_name in CLASS_NAMES:
    class_dir = DATA_ROOT / class_name
    files = sorted([p for p in class_dir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
    for p in files:
        all_candidates.append((p, class_name))

print(f"Checking integrity of {len(all_candidates):,} images (8 threads)...")
records = []
invalid = []

def check_img(item):
    p, c = item
    try:
        with Image.open(p) as im:
            im = ImageOps.exif_transpose(im)
            w, h = im.size
            mode = im.mode
            if (w, h) != (512, 512):
                return None, {"path": str(p), "reason": f"dimension_{w}x{h}"}
            im.convert("RGB").load()
        return {
            "path": str(p),
            "relative_path": str(p.relative_to(DATA_ROOT)),
            "class_name": c,
            "class_index": CLASS_TO_IDX[c],
            "width": w,
            "height": h,
            "mode": mode,
        }, None
    except Exception as e:
        return None, {"path": str(p), "reason": repr(e)}

t0_chk = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as ex:
    for i, (rec, inv) in enumerate(ex.map(check_img, all_candidates), 1):
        if rec: records.append(rec)
        if inv: invalid.append(inv)
        if i % 3000 == 0 or i == len(all_candidates):
            print(f"  [Checked {i:,} / {len(all_candidates):,} images ({time.time()-t0_chk:.1f}s)]")

raw_counts = {c: sum(r["class_name"] == c for r in records) for c in CLASS_NAMES}
print(f"Readable 512x512 images: {len(records)} | Invalid: {len(invalid)}")

# Save preprocessing validation report
prep_report = {
    "image_size": [512, 512],
    "readable_images": len(records),
    "invalid_count": len(invalid),
    "invalid_samples": invalid[:20],
    "class_counts": raw_counts
}
with open(ARTIFACT_DIR / "preprocessing_validation.json", "w") as f:
    json.dump(prep_report, f, indent=2)

# ==============================================================================
# 3. CANONICAL DIHEDRAL PHASH & CLUSTER SPLIT (RULES 2a-2e)
# ==============================================================================
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return
        if self.rank[ra] < self.rank[rb]: ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]: self.rank[ra] += 1

def compute_phash(img_path):
    with Image.open(img_path) as im0:
        im = ImageOps.exif_transpose(im0).convert('RGB')
        m = ImageOps.mirror(im)
        variants = [
            im, im.rotate(90, expand=True), im.rotate(180, expand=True), im.rotate(270, expand=True),
            m, m.rotate(90, expand=True), m.rotate(180, expand=True), m.rotate(270, expand=True),
        ]
        hashes = [imagehash.phash(v) for v in variants]
        return min(hashes, key=lambda h: h.hash.tobytes())

print(f"\nComputing canonical dihedral pHashes (8 threads)...")
t0_h = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as ex:
    for i, (rec, h) in enumerate(zip(records, ex.map(lambda r: compute_phash(r['path']), records)), 1):
        rec['phash'] = h
        if i % 3000 == 0 or i == len(records):
            print(f"  [Hashed {i:,} / {len(records):,} images ({time.time()-t0_h:.1f}s)]")

# Cross-class conflict check (Hamming <= 2 across DIFFERENT classes)
class_to_records = defaultdict(list)
for r in records:
    class_to_records[r["class_name"]].append(r)

conflicting_paths = set()
conflict_examples = defaultdict(list)
class_list = sorted(class_to_records.keys())

for i in range(len(class_list)):
    ca = class_list[i]
    for j in range(i + 1, len(class_list)):
        cb = class_list[j]
        for ra in class_to_records[ca]:
            for rb in class_to_records[cb]:
                dist = ra["phash"] - rb["phash"]
                if dist <= CONFLICT_MAX_DISTANCE:
                    conflicting_paths.add(ra["path"])
                    conflicting_paths.add(rb["path"])
                    if len(conflict_examples[ca]) < 10:
                        conflict_examples[ca].append({"this": ra["relative_path"], "other": rb["relative_path"], "other_class": cb, "dist": dist})
                    if len(conflict_examples[cb]) < 10:
                        conflict_examples[cb].append({"this": rb["relative_path"], "other": ra["relative_path"], "other_class": ca, "dist": dist})

print(f"Cross-class conflicting photos excluded: {len(conflicting_paths)}")

# Cluster WITHIN each class only
class_clusters = {}
class_cluster_counts = {}
for c in CLASS_NAMES:
    c_records = [r for r in class_to_records[c] if r["path"] not in conflicting_paths]
    n = len(c_records)
    uf = UnionFind(n)
    threshold = HAMMING_DIST_TOMATO if c.startswith("tomato_") else HAMMING_DIST_DEFAULT
    for ia in range(n):
        for ib in range(ia + 1, n):
            if c_records[ia]["phash"] - c_records[ib]["phash"] <= threshold:
                uf.union(ia, ib)
    clusters = defaultdict(list)
    for idx in range(n):
        root = uf.find(idx)
        clusters[root].append(c_records[idx])
    class_clusters[c] = list(clusters.values())
    class_cluster_counts[c] = len(class_clusters[c])

# Split 80/20 by cluster (0 cluster leakage)
rng = random.Random(SEED)
train_items, val_items = [], []
class_counts_v2 = {}
cluster_counter = 0

for c in CLASS_NAMES:
    clusters = list(class_clusters[c])
    cluster_count = len(clusters)
    source_n = raw_counts[c]
    conflict_n = sum(1 for r in class_to_records[c] if r["path"] in conflicting_paths)

    for cl in clusters:
        cid = f"{c}_cluster_{cluster_counter:05d}"
        cluster_counter += 1
        for item in cl:
            item["cluster_id"] = cid

    rng.shuffle(clusters)
    val_k = max(1, int(round(cluster_count * 0.20)))
    val_clusters = clusters[:val_k]
    train_clusters = clusters[val_k:]

    train_c = []
    for cl in train_clusters:
        train_c.extend(cl)

    val_c = []
    for cl in val_clusters:
        val_c.append(rng.choice(cl))

    if len(train_c) > MAX_TRAIN_PER_CLASS:
        rng.shuffle(train_c)
        train_c = train_c[:MAX_TRAIN_PER_CLASS]

    total_c = len(train_c) + len(val_c)
    if total_c > MAX_CLASS_TOTAL:
        excess = total_c - MAX_CLASS_TOTAL
        cut_train = min(excess, max(0, len(train_c) - 100))
        if cut_train > 0:
            train_c = train_c[:-cut_train]
        excess = (len(train_c) + len(val_c)) - MAX_CLASS_TOTAL
        if excess > 0:
            val_c = val_c[:-excess]

    train_items.extend(train_c)
    val_items.extend(val_c)

    train_n = len(train_c)
    val_n = len(val_c)
    kept_n = train_n + val_n
    same_dup_n = max(0, source_n - conflict_n - kept_n)
    is_small = (kept_n < 300) or (val_n < 30)

    class_counts_v2[c] = {
        "source_valid_512_count": source_n,
        "unique_clusters": cluster_count,
        "selected_unique_images_after_cap": kept_n,
        "train_count": train_n,
        "validation_count": val_n,
        "same_label_duplicate_files_removed": same_dup_n,
        "conflicting_label_files_excluded": conflict_n,
        "small_class": is_small,
        "cap": MAX_CLASS_TOTAL,
        "train_cap": MAX_TRAIN_PER_CLASS,
    }

rng.shuffle(train_items)
rng.shuffle(val_items)
train_groups = {x["cluster_id"] for x in train_items}
val_groups = {x["cluster_id"] for x in val_items}
overlap = sorted(train_groups & val_groups)
if overlap:
    raise RuntimeError(f"FATAL: Cluster leakage detected! Overlap: {overlap}")

for c in CLASS_NAMES:
    if class_counts_v2[c]["selected_unique_images_after_cap"] < 300:
        raise RuntimeError(f"Class {c} dropped below 300 photos!")

print(f"\n✓ Dataset Split Validated: {len(train_items):,} train photos | {len(val_items):,} val photos")
print("  Zero cluster leakage confirmed!")

# Save split artifacts (support both standard and alias filenames)
with open(ARTIFACT_DIR / "class_counts_v2.json", "w") as f:
    json.dump(class_counts_v2, f, indent=2)
with open(ARTIFACT_DIR / "class_counts.json", "w") as f:
    json.dump(class_counts_v2, f, indent=2)
with open(ARTIFACT_DIR / "split_v2.json", "w") as f:
    json.dump({"train": train_items, "validation": val_items}, f, indent=2)

leakage_payload = {
    "train_samples": len(train_items),
    "validation_samples": len(val_items),
    "zero_cluster_leakage": True,
    "cluster_overlap_count": 0
}
with open(ARTIFACT_DIR / "leakage_after.json", "w") as f:
    json.dump(leakage_payload, f, indent=2)
with open(ARTIFACT_DIR / "leakage_report.json", "w") as f:
    json.dump(leakage_payload, f, indent=2)

# ==============================================================================
# 4. DATA LOADERS & DATA PARALLEL MODEL
# ==============================================================================
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(512, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.RandomRotation(20, fill=(114, 114, 114)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class CropDataset(Dataset):
    def __init__(self, items, transform):
        self.items = items
        self.transform = transform
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        row = self.items[idx]
        with Image.open(row["path"]) as im:
            image = ImageOps.exif_transpose(im).convert("RGB")
        return self.transform(image), int(row["class_index"])

train_ds = CropDataset(train_items, train_transform)
val_ds = CropDataset(val_items, val_transform)

train_targets = [int(item["class_index"]) for item in train_items]
train_counter = Counter(train_targets)
sample_weights = [1.0 / train_counter[t] for t in train_targets]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(train_items), replacement=True)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
    persistent_workers=True if NUM_WORKERS > 0 else False
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
    persistent_workers=True if NUM_WORKERS > 0 else False
)

weights = models.MobileNet_V3_Large_Weights.DEFAULT
base_model = models.mobilenet_v3_large(weights=weights)
base_model.classifier[3] = nn.Linear(base_model.classifier[3].in_features, len(CLASS_NAMES))
base_model = base_model.to(device)

if gpu_count > 1:
    print(f"Enabling DataParallel across all {gpu_count} GPUs!")
    model = nn.DataParallel(base_model)
else:
    model = base_model

counts = torch.tensor([max(1, class_counts_v2[c]["train_count"]) for c in CLASS_NAMES], dtype=torch.float32)
class_weights = (1.0 / torch.sqrt(counts))
class_weights = (class_weights / class_weights.mean()).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

# ==============================================================================
# 5. RESUMABLE 30-EPOCH TRAINING LOOP WITH EARLY STOPPING (TASK TR2)
# ==============================================================================
BEST_MODEL_PATH = ARTIFACT_DIR / "best_model.pth"
HISTORY_PATH = ARTIFACT_DIR / "training_history.json"
WORKING_BEST_CKPT = WORKING_DIR / "checkpoint_best.pt"

start_epoch = 0
best_val_macro_acc = 0.0
best_epoch = 0
epochs_without_improvement = 0
history = []

# Check for existing checkpoints in /kaggle/working to auto-resume
ckpt_candidates = list(WORKING_DIR.glob("checkpoint_epoch_*.pt"))
if not ckpt_candidates and CHECKPOINT_DIR.exists():
    ckpt_candidates = list(CHECKPOINT_DIR.glob("checkpoint_epoch_*.pt"))

if ckpt_candidates:
    def parse_ep(p):
        try:
            return int(p.stem.split("_")[-1])
        except Exception:
            return -1
    latest_ckpt_path = max(ckpt_candidates, key=parse_ep)
    resume_ep = parse_ep(latest_ckpt_path)
    if resume_ep > 0 and resume_ep < EPOCHS:
        print(f"\n>>> Resuming training from checkpoint: {latest_ckpt_path} (Completed Epoch {resume_ep})")
        ckpt = torch.load(latest_ckpt_path, map_location=device, weights_only=False)
        if hasattr(model, "module"):
            model.module.load_state_dict(ckpt["model_state"])
        else:
            model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        if "scaler_state" in ckpt and torch.cuda.is_available():
            scaler.load_state_dict(ckpt["scaler_state"])
        start_epoch = ckpt["epoch"]
        best_val_macro_acc = ckpt.get("best_val_macro_acc", 0.0)
        best_epoch = ckpt.get("best_epoch", 0)
        epochs_without_improvement = ckpt.get("epochs_without_improvement", 0)
        history = ckpt.get("history", [])
        print(f"✓ Resumed state: start_epoch={start_epoch}, best_epoch={best_epoch}, best_macro_acc={best_val_macro_acc*100:.2f}%")

print(f"\n{'='*80}")
print(f"STARTING 30-EPOCH HIGH-THROUGHPUT TRAINING (BATCH SIZE: {BATCH_SIZE})")
print(f"Train batches per epoch: {len(train_loader)} | Val batches: {len(val_loader)}")
print(f"Early stopping patience: {PATIENCE} epochs on grouped validation macro accuracy")
print(f"{'='*80}\n")

for epoch in range(start_epoch, EPOCHS):
    t0_ep = time.time()
    
    # Train
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    t_batch = time.time()
    
    for b_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            logits = model(images)
            loss = criterion(logits, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        train_loss += loss.item() * labels.size(0)
        train_correct += (logits.argmax(1) == labels).sum().item()
        train_total += labels.size(0)
        
        if (b_idx + 1) % 50 == 0 or (b_idx + 1) == len(train_loader):
            elapsed = time.time() - t_batch
            print(f"  [Epoch {epoch+1:02d}/{EPOCHS} | Batch {b_idx+1:03d}/{len(train_loader)}] "
                  f"loss: {train_loss/train_total:.4f} acc: {train_correct/train_total:.4f} ({elapsed:.1f}s)")
            
    train_loss_epoch = train_loss / train_total
    train_acc_epoch = train_correct / train_total
    
    # Grouped Validation
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    val_preds_list, val_targets_list = [], []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                logits = model(images)
                loss = criterion(logits, labels)
            val_loss += loss.item() * labels.size(0)
            preds = logits.argmax(1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
            val_preds_list.append(preds.cpu())
            val_targets_list.append(labels.cpu())
            
    val_loss_epoch = val_loss / val_total
    val_acc_epoch = val_correct / val_total
    scheduler.step()
    
    # Calculate Grouped Validation Macro Accuracy (unweighted mean of per-class recalls)
    val_preds_all = torch.cat(val_preds_list, dim=0)
    val_targets_all = torch.cat(val_targets_list, dim=0)
    per_class_recalls_ep = []
    for c_idx in range(len(CLASS_NAMES)):
        c_mask = (val_targets_all == c_idx)
        if c_mask.sum() > 0:
            c_acc = float((val_preds_all[c_mask] == c_idx).float().mean().item())
            per_class_recalls_ep.append(c_acc)
        else:
            per_class_recalls_ep.append(0.0)
    val_macro_acc = float(np.mean(per_class_recalls_ep))
    
    ep_duration = time.time() - t0_ep
    print(f"\n>>> EPOCH {epoch+1:02d}/{EPOCHS} RESULT | train_loss: {train_loss_epoch:.4f} acc: {train_acc_epoch:.4f} | "
          f"val_loss: {val_loss_epoch:.4f} overall_acc: {val_acc_epoch:.4f} macro_acc: {val_macro_acc:.4f} | Time: {ep_duration:.1f}s\n")
    
    saved_state = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
    
    # Check best model by grouped validation macro accuracy
    if val_macro_acc > best_val_macro_acc:
        best_val_macro_acc = val_macro_acc
        best_epoch = epoch + 1
        epochs_without_improvement = 0
        best_payload = {
            "epoch": best_epoch,
            "model_state": saved_state,
            "class_names": CLASS_NAMES,
            "input_size": 512,
            "best_epoch": best_epoch,
            "best_val_macro_acc": best_val_macro_acc,
            "val_overall_acc": val_acc_epoch,
            "val_loss": val_loss_epoch
        }
        torch.save(best_payload, BEST_MODEL_PATH)
        torch.save(best_payload, WORKING_BEST_CKPT)
        print(f"  ★ New best model checkpoint saved: Macro Acc = {best_val_macro_acc*100:.2f}% (Epoch {best_epoch})")
    else:
        epochs_without_improvement += 1
        print(f"  No macro acc improvement for {epochs_without_improvement} epoch(s) (Best: {best_val_macro_acc*100:.2f}% at epoch {best_epoch})")
    
    # Save checkpoint to /kaggle/working every epoch for auto-resume
    epoch_ckpt_path = WORKING_DIR / f"checkpoint_epoch_{epoch+1:02d}.pt"
    ckpt_state = {
        "epoch": epoch + 1,
        "model_state": saved_state,
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict() if torch.cuda.is_available() else {},
        "val_macro_acc": val_macro_acc,
        "val_overall_acc": val_acc_epoch,
        "val_loss": val_loss_epoch,
        "best_val_macro_acc": best_val_macro_acc,
        "best_epoch": best_epoch,
        "epochs_without_improvement": epochs_without_improvement,
        "history": history,
        "class_names": CLASS_NAMES,
    }
    torch.save(ckpt_state, epoch_ckpt_path)
    torch.save(ckpt_state, WORKING_DIR / "checkpoint_latest.pt")
    if CHECKPOINT_DIR != WORKING_DIR:
        torch.save(ckpt_state, CHECKPOINT_DIR / f"checkpoint_epoch_{epoch+1:02d}.pt")

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss_epoch,
        "train_accuracy": train_acc_epoch,
        "validation_loss": val_loss_epoch,
        "validation_accuracy": val_acc_epoch,
        "validation_macro_accuracy": val_macro_acc,
        "learning_rate": optimizer.param_groups[0]["lr"],
        "seconds": ep_duration
    })
    with open(HISTORY_PATH, "w") as f:
        json.dump(history, f, indent=2)
    with open(WORKING_DIR / "training_history.json", "w") as f:
        json.dump(history, f, indent=2)

    # Early stopping check
    if epochs_without_improvement >= PATIENCE:
        print(f"\n{'='*80}")
        print(f"⚠ EARLY STOPPING TRIGGERED at Epoch {epoch+1:02d}!")
        print(f"  No improvement in grouped validation macro accuracy for {PATIENCE} consecutive epochs.")
        print(f"  Best checkpoint restored from Epoch {best_epoch:02d} with Macro Accuracy: {best_val_macro_acc*100:.2f}%")
        print(f"{'='*80}\n")
        break

print(f"\n✓ Training completed! Best epoch: {best_epoch} with Macro Accuracy: {best_val_macro_acc*100:.2f}%")

# ==============================================================================
# 6. CALIBRATION & EVALUATION ARTIFACTS
# ==============================================================================
print("\nComputing temperature calibration & validation confusion matrix...")
best_ckpt = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=False)
eval_model = models.mobilenet_v3_large(weights=None)
eval_model.classifier[3] = nn.Linear(eval_model.classifier[3].in_features, len(CLASS_NAMES))
eval_model.load_state_dict(best_ckpt["model_state"])
eval_model = eval_model.to(device).eval()

all_logits, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            all_logits.append(eval_model(images).float().cpu())
        all_labels.append(labels.cpu())

val_logits = torch.cat(all_logits, dim=0)
val_labels = torch.cat(all_labels, dim=0)

# Uncalibrated NLL
loss_criterion = nn.CrossEntropyLoss()
nll_before = float(loss_criterion(val_logits, val_labels).item())

# Temperature optimization
temp_param = nn.Parameter(torch.ones(1) * 1.5)
temp_opt = torch.optim.LBFGS([temp_param], lr=0.01, max_iter=50)
def eval_temp():
    temp_opt.zero_grad()
    loss = nn.CrossEntropyLoss()(val_logits / temp_param.clamp(min=0.1), val_labels)
    loss.backward()
    return loss
temp_opt.step(eval_temp)
T = float(temp_param.clamp(min=0.1).item())
scaled_logits = val_logits / T
nll_after = float(loss_criterion(scaled_logits, val_labels).item())
print(f"✓ Calibrated Temperature T: {T:.4f} (NLL: {nll_before:.4f} -> {nll_after:.4f})")

# Energy threshold at 95% TPR
free_energies = -T * torch.logsumexp(scaled_logits, dim=1)
energy_thresh = float(torch.quantile(free_energies, 0.95).item())
probs = torch.softmax(scaled_logits, dim=1)
max_probs, preds = probs.max(dim=1)
conf_thresh = float(torch.quantile(max_probs, 0.05).item())

# Per-class grouped metrics
per_class_stats = {}
per_class_acc = {}
for i, c in enumerate(CLASS_NAMES):
    mask = (val_labels == i)
    support = int(mask.sum().item())
    correct = int((preds[mask] == i).sum().item())
    acc = float(correct / support) if support > 0 else 0.0
    pred_mask = (preds == i)
    pred_count = int(pred_mask.sum().item())
    precision = float(correct / pred_count) if pred_count > 0 else 0.0
    recall = acc
    f1 = float(2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    
    per_class_acc[c] = acc
    per_class_stats[c] = {
        "accuracy_recall": acc,
        "correct": correct,
        "support": support,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }

macro_acc = float(np.mean(list(per_class_acc.values())))
overall_acc = float((preds == val_labels).float().mean().item())

thresholds_payload = {
    "temperature": round(T, 4),
    "confidence_threshold": round(conf_thresh, 4),
    "energy_threshold": round(energy_thresh, 4),
    "energy_method": "Free Energy score E(x) = -T * log(sum(exp(z_i / T))) at 95% TPR on grouped validation set",
    "status": "PASS",
    "macro_validation_accuracy": round(macro_acc, 6),
    "overall_validation_accuracy": round(overall_acc, 6),
    "per_class_accuracy": {c: round(per_class_acc[c], 4) for c in CLASS_NAMES}
}
with open(ARTIFACT_DIR / "thresholds.json", "w") as f:
    json.dump(thresholds_payload, f, indent=2)

# Confusion matrix
cm = np.zeros((len(CLASS_NAMES), len(CLASS_NAMES)), dtype=int)
for t, p in zip(val_labels.numpy(), preds.numpy()):
    cm[t, p] += 1
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
cm_df.to_csv(ARTIFACT_DIR / "confusion_matrix.csv")

# Extract top confusions
top_confusions = []
for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        if i != j and cm[i, j] > 0:
            top_confusions.append({
                "true_class": CLASS_NAMES[i],
                "pred_class": CLASS_NAMES[j],
                "count": int(cm[i, j])
            })
top_confusions.sort(key=lambda x: x["count"], reverse=True)

# Save labels
with open(ARTIFACT_DIR / "labels.json", "w") as f:
    json.dump(CLASS_NAMES, f, indent=2)

metrics_payload = {
    "model": "MobileNetV3-Large",
    "input_shape": [1, 3, 512, 512],
    "output_shape": [1, 20],
    "class_order": CLASS_NAMES,
    "train_images": len(train_items),
    "validation_images": len(val_items),
    "epochs_completed": len(history),
    "best_epoch": best_epoch,
    "overall_accuracy": overall_acc,
    "macro_accuracy_balanced_accuracy": macro_acc,
    "macro_accuracy": macro_acc,
    "per_class_grouped_accuracy": per_class_stats,
    "per_class_accuracy": {c: round(per_class_acc[c], 4) for c in CLASS_NAMES},
    "top_confusions": top_confusions[:15],
    "calibration_method": "single_scalar_temperature_scaling",
    "temperature": T,
    "confidence_threshold": conf_thresh,
    "energy_threshold": energy_thresh,
    "energy_method": "free_energy_validation_95th_percentile",
    "validation_nll_before_temperature_scaling": nll_before,
    "validation_nll_after_temperature_scaling": nll_after,
    "model_file_size_mb": None,
    "rule_h15_budget_mb": 30.0,
    "rule_h15_compliance": None,
    "versions": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "torchvision": torchvision.__version__,
        "executorch": APP_REQUIRED_EXECUTORCH
    }
}
with open(ARTIFACT_DIR / "metrics.json", "w") as f:
    json.dump(metrics_payload, f, indent=2)

# Save preprocess.json
preprocess_payload = {
    "input_size": 512,
    "mean": [0.485, 0.456, 0.406],
    "std": [0.229, 0.224, 0.225],
    "temperature": round(T, 4),
    "confidence_threshold": round(conf_thresh, 4),
    "energy_threshold": round(energy_thresh, 4),
    "energy_method": "free_energy_validation_95th_percentile",
    "model_version": "20261121_512_30ep",
    "model_bytes": None
}
with open(ARTIFACT_DIR / "preprocess.json", "w") as f:
    json.dump(preprocess_payload, f, indent=2)

print(f"✓ Overall Validation Accuracy: {overall_acc*100:.2f}%")
print(f"✓ Macro Validation Accuracy:   {macro_acc*100:.2f}%")

# ==============================================================================
# 7. AUTOMATED EXECUTORCH 0.6.0 EXPORT & 10-SAMPLE PARITY TEST
# ==============================================================================
print("\n" + "=" * 80)
print(f"EXPORTING TO EXECUTORCH {APP_REQUIRED_EXECUTORCH} XNNPACK (.pte)")
print("=" * 80)

# If running on Python >= 3.13, auto-setup Python 3.11 export venv
if sys.version_info >= (3, 13):
    print("Detected Python >= 3.13. Setting up automated Python 3.11 environment via uv...")
    os.system("pip install -q uv")
    os.system("uv venv --python 3.11 /tmp/py311")
    os.system('uv pip install --python /tmp/py311 --no-deps "https://download.pytorch.org/whl/cpu/executorch-0.6.0%2Bcpu-cp311-cp311-manylinux_2_28_x86_64.whl"')
    os.system("uv pip install --python /tmp/py311 ruamel.yaml tabulate packaging sympy flatbuffers")
    os.system("uv pip install --python /tmp/py311 --extra-index-url https://download.pytorch.org/whl/cpu torch torchvision")
    EXPORT_PY = "/tmp/py311/bin/python"
else:
    os.system("pip install -q executorch==0.6.0")
    EXPORT_PY = sys.executable

# Export and parity script written to a dedicated file
export_script_content = """import sys, json, random
from pathlib import Path
from PIL import Image, ImageOps
import torch
from torch import nn
from torchvision import models, transforms

RUN_DIR = Path("__RUN_DIR__")
ARTIFACT_DIR = Path("__ARTIFACT_DIR__")
BEST_MODEL_PATH = Path("__BEST_MODEL_PATH__")
PTE_PATH = ARTIFACT_DIR / "vision.pte"
CLASS_NAMES = __CLASS_NAMES__
SEED = __SEED__

print("Running ExecuTorch export with Python:", sys.version)
import importlib.metadata as meta
try:
    et_ver = meta.version("executorch")
except Exception:
    et_ver = "0.6.0"
print("ExecuTorch installed:", et_ver)

from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.exir import to_edge_transform_and_lower

# Load model weights
model_export = models.mobilenet_v3_large(weights=None)
model_export.classifier[3] = nn.Linear(model_export.classifier[3].in_features, len(CLASS_NAMES))
ckpt = torch.load(BEST_MODEL_PATH, map_location="cpu", weights_only=False)
model_export.load_state_dict(ckpt["model_state"])
model_export.eval()

# Export program
example = torch.randn(1, 3, 512, 512, dtype=torch.float32)
exported = torch.export.export(model_export, (example,))
edge_prog = to_edge_transform_and_lower(exported, partitioner=[XnnpackPartitioner()])
et_prog = edge_prog.to_executorch()

with open(PTE_PATH, "wb") as f:
    f.write(et_prog.buffer)

pte_size_bytes = PTE_PATH.stat().st_size
pte_size_mb = pte_size_bytes / (1024 * 1024)
print(f"✓ Successfully exported vision.pte: {pte_size_mb:.2f} MB (Rule H15 budget < 30MB: {pte_size_mb < 30.0})")

# 10-Sample Parity Check on Real Grouped Validation Photos
val_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

with open(ARTIFACT_DIR / "split_v2.json") as f:
    val_items = json.load(f)["validation"]

parity_rows = random.Random(SEED).sample(val_items, k=min(10, len(val_items)))

try:
    from executorch.runtime import Runtime
    runtime = Runtime.get()
    prog_rt = runtime.load_program(str(PTE_PATH))
    method = prog_rt.load_method("forward")
    
    parity_results = []
    for row in parity_rows:
        with Image.open(row["path"]) as im:
            image = ImageOps.exif_transpose(im).convert("RGB")
        x = val_transform(image).unsqueeze(0).float()
        with torch.no_grad():
            py_out = model_export(x).cpu()
        et_out = method.execute([x.contiguous()])[0]
        if not isinstance(et_out, torch.Tensor):
            et_out = torch.tensor(et_out)
        abs_diff = (py_out - et_out).abs()
        max_abs = float(abs_diff.max().item())
        mean_abs = float(abs_diff.mean().item())
        parity_results.append({
            "relative_path": row.get("relative_path", str(row["path"])),
            "true_class": row["class_name"],
            "max_absolute_error": max_abs,
            "mean_absolute_error": mean_abs,
            "pass": max_abs < 1e-3
        })
    
    overall_max_err = max(r["max_absolute_error"] for r in parity_results)
    parity_payload = {
        "backend": "XNNPACK",
        "input_shape": [1, 3, 512, 512],
        "output_shape": [1, 20],
        "tolerance": 0.001,
        "samples_tested": len(parity_results),
        "max_error_across_all_samples": overall_max_err,
        "all_pass": overall_max_err < 1e-3,
        "results": parity_results
    }
except Exception as e:
    parity_payload = {
        "backend": "XNNPACK",
        "tolerance": 0.001,
        "status": "EXCEPTION",
        "error": str(e),
        "all_pass": False
    }

with open(ARTIFACT_DIR / "parity_samples.json", "w") as f:
    json.dump(parity_payload, f, indent=2)

print("✓ Parity test on 10 validation photos completed:", parity_payload.get("all_pass"))
"""

export_script_filled = export_script_content \
    .replace("__RUN_DIR__", str(RUN_DIR).replace("\\", "/")) \
    .replace("__ARTIFACT_DIR__", str(ARTIFACT_DIR).replace("\\", "/")) \
    .replace("__BEST_MODEL_PATH__", str(BEST_MODEL_PATH).replace("\\", "/")) \
    .replace("__CLASS_NAMES__", json.dumps(CLASS_NAMES)) \
    .replace("__SEED__", str(SEED))

runner_file = RUN_DIR / "run_export.py"
with open(runner_file, "w") as f:
    f.write(export_script_filled)

os.system(f"{EXPORT_PY} {runner_file}")

PTE_PATH = ARTIFACT_DIR / "vision.pte"
assert PTE_PATH.exists(), "Fatal: vision.pte was not created!"
pte_size_bytes = PTE_PATH.stat().st_size
pte_size_mb = pte_size_bytes / (1024 * 1024)

# Update metrics.json and preprocess.json with model size
with open(ARTIFACT_DIR / "metrics.json") as f:
    m_data = json.load(f)
m_data["model_file_size_mb"] = round(pte_size_mb, 3)
m_data["rule_h15_compliance"] = "PASS" if pte_size_mb < 30.0 else "FAIL"
with open(ARTIFACT_DIR / "metrics.json", "w") as f:
    json.dump(m_data, f, indent=2)

with open(ARTIFACT_DIR / "preprocess.json") as f:
    p_data = json.load(f)
p_data["model_bytes"] = pte_size_bytes
with open(ARTIFACT_DIR / "preprocess.json", "w") as f:
    json.dump(p_data, f, indent=2)

# Load parity samples results
with open(ARTIFACT_DIR / "parity_samples.json") as f:
    parity_data = json.load(f)

# ==============================================================================
# 8. COMPARISON WITH 15-EPOCH BASELINE & SWAP GATES EVALUATION (TASK TR2)
# ==============================================================================
b_overall = BASELINE_METRICS["overall_accuracy"]
b_macro = BASELINE_METRICS["macro_accuracy"]

gate_overall_pass = (overall_acc >= 0.9149)
gate_macro_pass = (macro_acc >= 0.9217)

parity_max_err = parity_data.get("max_error_across_all_samples", 0.0)
gate_parity_pass = (parity_max_err < 1e-3) if parity_data.get("all_pass") else False

# Check each class drop
class_drops = {}
gate_classes_pass = True
for c in CLASS_NAMES:
    b_c = BASELINE_METRICS["classes"][c]
    n_c = per_class_acc[c]
    diff = n_c - b_c
    drop = b_c - n_c
    c_pass = (drop <= 0.0300 + 1e-6)
    if not c_pass:
        gate_classes_pass = False
    class_drops[c] = {
        "baseline": b_c,
        "new": n_c,
        "diff": diff,
        "pass": c_pass
    }

all_gates_pass = gate_overall_pass and gate_macro_pass and gate_classes_pass and gate_parity_pass

print("\n" + "=" * 90)
print("             CROP DOCTOR MODEL SWAP GATES EVALUATION: 15-EP vs 30-EP")
print("=" * 90)
print(f"{'Metric / Target':<35} | {'15-Epoch':<12} | {'30-Epoch':<12} | {'Diff':<10} | {'Swap Gate'}")
print("-" * 90)
print(f"{'Overall Accuracy (>= 91.49%)':<35} | {b_overall*100:6.2f}%      | {overall_acc*100:6.2f}%      | {(overall_acc-b_overall)*100:+5.2f}%    | {'PASS' if gate_overall_pass else 'FAIL'}")
print(f"{'Macro Accuracy   (>= 92.17%)':<35} | {b_macro*100:6.2f}%      | {macro_acc*100:6.2f}%      | {(macro_acc-b_macro)*100:+5.2f}%    | {'PASS' if gate_macro_pass else 'FAIL'}")
print(f"{'Parity Max Error (< 1e-3)':<35} | {'< 1e-3':<12} | {parity_max_err:<12.6f} | {'N/A':<10} | {'PASS' if gate_parity_pass else 'FAIL'}")
print(f"{'Model Size (< 30 MB)':<35} | {'16.17 MB':<12} | {f'{pte_size_mb:.2f} MB':<12} | {'N/A':<10} | {'PASS' if pte_size_mb < 30.0 else 'FAIL'}")
print("-" * 90)
print(f"{'Per-Class Accuracy (No class drop > 3.0 percentage points):':<35}")
for c in CLASS_NAMES:
    cd = class_drops[c]
    c_status = "PASS" if cd["pass"] else "FAIL (DROP > 3%)"
    print(f"  {c:<33} | {cd['baseline']*100:6.2f}%      | {cd['new']*100:6.2f}%      | {cd['diff']*100:+5.2f}%    | {c_status}")
print("=" * 90)

if all_gates_pass:
    print("🏆 OVERALL SWAP GATE VERDICT: PASS — NEW 30-EPOCH MODEL QUALIFIES FOR APP DEPLOYMENT!")
else:
    print("⚠ OVERALL SWAP GATE VERDICT: FAIL — ONE OR MORE GATES DID NOT PASS. KEEP 15-EPOCH MODEL.")
print("=" * 90)

# ==============================================================================
# 9. BUNDLE FINAL DELIVERABLES ZIP (TASK TR2)
# ==============================================================================
files_to_pack = [
    ("vision.pte", "vision.pte"),
    ("labels.json", "labels.json"),
    ("thresholds.json", "thresholds.json"),
    ("metrics.json", "metrics.json"),
    ("class_counts.json", "class_counts.json"),
    ("class_counts_v2.json", "class_counts_v2.json"),
    ("leakage_report.json", "leakage_report.json"),
    ("leakage_after.json", "leakage_after.json"),
    ("preprocess.json", "preprocess.json"),
    ("preprocessing_validation.json", "preprocessing_validation.json"),
    ("parity_samples.json", "parity_samples.json"),
    ("training_history.json", "training_history.json"),
    ("confusion_matrix.csv", "confusion_matrix.csv"),
    ("split_v2.json", "split_v2.json"),
    ("best_model.pth", "best_model.pth"),
]

with zipfile.ZipFile(FINAL_ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for src_name, arc_name in files_to_pack:
        p = ARTIFACT_DIR / src_name
        if p.exists():
            zf.write(p, arcname=arc_name)

print("\n" + "=" * 80)
print(f"🎉 SUCCESS! All Task TR2 artifacts bundled into:")
print(f"   {FINAL_ZIP_PATH} ({FINAL_ZIP_PATH.stat().st_size / (1024*1024):.2f} MB)")
print("=" * 80)
print("👉 IN KAGGLE: Look at the right sidebar under Output -> /kaggle/working/")
print("   Click the three dots (...) next to 'crop_doctor_final_30ep.zip' and click Download!")
print("=" * 80)
